# QSV2G — modular Kaggle experiment runner
This notebook installs the **same** modular package used by the CLI. CPU and Internet are required for first-time setup.
Quantum operations are software simulations; real ML-KEM/ML-DSA is used where supported.


In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = 'https://github.com/shanmukhdatta/QSV2G.git'
SOURCE_REVISION = 'f66f026ffbe7abf5157728c6d47d43346d849732'  # Verified modular source
STUDIES = ['validation','security','security-stateful','performance','noise','noise-vector','availability','security-adversarial','reliability']
FULL = False  # Choose True only for the much larger research workload.
ROOT = Path('/kaggle/working/QSV2G')
OUTPUT_DIR = Path('/kaggle/working/qsv2g_new_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not ROOT.exists(): subprocess.run(['git','clone',REPO,str(ROOT)],check=True)
subprocess.run(['git','fetch','--quiet','origin'],cwd=ROOT,check=True)
subprocess.run(['git','checkout','--detach',SOURCE_REVISION],cwd=ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev,pqc]'],cwd=ROOT,check=True)
tests = subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=ROOT,capture_output=True,text=True)
(OUTPUT_DIR/'pytest.log').write_text(tests.stdout+'\n'+tests.stderr)
print('Test outcome:',tests.returncode,tests.stdout[-400:])
if tests.returncode: raise RuntimeError('Regression tests failed; inspect pytest.log')
print('Pinned source revision:',SOURCE_REVISION)


In [ ]:
for study in STUDIES:
    command = [sys.executable,'scripts/run_experiment.py','--study',study,'--out',str(OUTPUT_DIR)]
    if FULL: command.append('--full')
    subprocess.run(command,cwd=ROOT,check=True)
print('Completed selected studies:',len(STUDIES))


In [ ]:
import zipfile, hashlib, json
from datetime import datetime, timezone
out = Path('/kaggle/working/qsv2g_modular_results.zip')
paths = sorted(p for p in OUTPUT_DIR.rglob('*') if p.is_file())
hashes = {str(p.relative_to(OUTPUT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
manifest = {'source_revision':SOURCE_REVISION,'full_mode':FULL,'studies':STUDIES,
            'created_utc':datetime.now(timezone.utc).isoformat(),'sha256':hashes}
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in paths: z.write(p,arcname=Path('results')/p.relative_to(OUTPUT_DIR))
    z.writestr('manifest_sha256.json',json.dumps(manifest,indent=2))
print(out,len(paths),'new-run artifacts; historical results excluded')
